# Notebook 01: Data Collection

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li  

## Overview

This notebook handles data acquisition. The primary dataset is the Boston Building and Property Violations dataset, which is publicly available via Boston's Open Data portal. We also document the neighborhood population data sourced from the U.S. Census Bureau, which will be used in Notebook 04 for normalization.

**Data sources:**
- Boston Building & Property Violations: https://data.boston.gov/
- Boston Population Estimates: https://data.boston.gov/dataset/2025-boston-population-estimates-neighborhood-level

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# Project root relative to this notebook
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure verified.')
print(f'Raw data dir: {RAW_DIR}')

## 1. Load the Violations Dataset

The raw CSV was downloaded from the Boston Open Data portal and placed in `data/raw/violations.csv`. We load and inspect it here.

In [ ]:
violations_path = RAW_DIR / 'violations.csv'
df = pd.read_csv(violations_path, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Initial Data Inspection

We check column data types, missing value rates, and the date range covered by the dataset.

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values (%) ---')
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0])

In [ ]:
# Parse the status date column
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
valid_dates = df['status_dttm'].dropna()
print(f'Date range: {valid_dates.min()} → {valid_dates.max()}')
print(f'Records with valid date: {valid_dates.shape[0]:,} of {len(df):,}')

## 3. Neighborhood Population Data (External)

We use official neighborhood population estimates from the City of Boston 
Planning Department Research Division. The data is downloaded from the 
Boston Open Data Portal (data.boston.gov)

In [ ]:
population_path = EXTERNAL_DIR / 'boston_population_estimates_2025_neighborhood_level.csv'

if not population_path.exists():
    raise FileNotFoundError(
        f'Missing population file: {population_path}. '
        'Download it from the Boston 2025 neighborhood population estimates page '
        'and save it in data/external/.'
    )

pop_raw = pd.read_csv(population_path)
pop_raw.columns = pop_raw.columns.str.strip()

name_col = next((c for c in ['name', 'neighborhood', 'Neighborhood'] if c in pop_raw.columns), None)
population_candidates = [
    'population_b03002_001e',  # total population from race/ethnicity table
    'population_b01001_001e',  # total population from age/sex table
]
pop_col = next((c for c in population_candidates if c in pop_raw.columns), None)

if name_col is None:
    raise KeyError(f'Could not find a neighborhood name column. Available columns: {pop_raw.columns.tolist()}')

if pop_col is None:
    possible_cols = [c for c in pop_raw.columns if c.lower().startswith('population')]
    raise KeyError(
        'Could not find the expected total population column. '
        f'Population-like columns include: {possible_cols[:15]}'
    )

pop_df = pop_raw[[name_col, pop_col]].copy()
pop_df = pop_df.rename(columns={name_col: 'neighborhood', pop_col: 'population_2025'})
pop_df['population_2025'] = pd.to_numeric(pop_df['population_2025'], errors='coerce')
pop_df = pop_df.dropna(subset=['neighborhood', 'population_2025'])
pop_df['neighborhood'] = pop_df['neighborhood'].astype(str).str.strip()
pop_df['population_2025'] = pop_df['population_2025'].round().astype(int)
pop_df = pop_df.sort_values('neighborhood').reset_index(drop=True)

pop_df.to_csv(EXTERNAL_DIR / 'neighborhood_population.csv', index=False)
print(f'Loaded population from: {population_path.name}')
print(f'Using columns: {name_col!r}, {pop_col!r}')
print(f'Saved {len(pop_df)} neighborhood population rows.')
pop_df.head()

## 4. Unique Values in Key Columns

We examine the range of violation codes, statuses, and cities to understand what the dataset covers.

In [ ]:
print('Unique statuses:', df['status'].unique())
print(f'Unique violation codes: {df["code"].nunique()}')
print(f'Unique violation descriptions: {df["description"].nunique()}')
print('Top 10 cities in data:')
print(df['violation_city'].value_counts().head(10))

## 5. Property Assessment Data (External)

To add structural housing context, we include Boston Property Assessment data from the Boston Assessing Department / Analyze Boston. This dataset provides parcel-level property characteristics such as property type, year built, living area, land area, and assessed value.

The raw CSV is large, so it is kept locally as:

```text
data/external/boston_property_assessment_data.csv
```

For GitHub reproducibility, the processed neighborhood-level feature table is saved as:

```text
data/external/neighborhood_structural_features.csv
```

The raw file is intentionally not committed because it is larger than GitHub's normal single-file size limit. Notebook 02 documents and implements the cleaning and aggregation step.


In [ ]:
property_path = EXTERNAL_DIR / 'boston_property_assessment_data.csv'
structural_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'

if property_path.exists():
    prop_sample = pd.read_csv(property_path, nrows=5, low_memory=False)
    print(f'Property assessment raw file found: {property_path}')
    print(f'Columns: {len(prop_sample.columns)}')
    print(prop_sample.columns.tolist())
    display(prop_sample.head())
else:
    print('Raw property assessment CSV not found locally.')
    print('Expected path:', property_path)
    print('Using processed structural feature file if available:', structural_path.exists())

if structural_path.exists():
    structural_sample = pd.read_csv(structural_path)
    print(f'Processed structural feature rows: {len(structural_sample)}')
    display(structural_sample.head())


## 5. Summary

Key findings from data collection:
- The dataset contains **~17,000 violation records** spanning multiple years.
- Key fields include violation code, description, city/neighborhood, status, and geo-coordinates.
- A meaningful fraction of records lack a `status_dttm` (date), which we address in Notebook 02.
- External population data has been saved to `data/external/neighborhood_population.csv` for later normalization.

In [ ]:
# Save a summary metadata file
summary = {
    'total_records': int(len(df)),
    'columns': df.columns.tolist(),
    'date_min': str(valid_dates.min()),
    'date_max': str(valid_dates.max()),
    'unique_codes': int(df['code'].nunique()),
}
with open(PROCESSED_DIR / 'dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to data/processed/dataset_summary.json')